In [ ]:
import nltk
import random
from nltk.corpus import movie_reviews

def extract_features(words):
    return dict([(word, True) for word in words])

def text_classification():
    nltk.download('movie_reviews', quiet=True)

    documents = [(list(movie_reviews.words(fileid)), category)
                 for category in movie_reviews.categories()
                 for fileid in movie_reviews.fileids(category)]

    random.shuffle(documents)

    featuresets = [(extract_features(words), category) for (words, category) in documents]
    train_set, test_set = featuresets[:1500], featuresets[1500:]

    classifier = nltk.NaiveBayesClassifier.train(train_set)

    accuracy = nltk.classify.accuracy(classifier, test_set)
    print("Accuracy:", accuracy)

    user_input = input("Enter a movie review: ")
    features = extract_features(user_input.split())

    sentiment = classifier.classify(features)
    print("Sentiment:", sentiment)

if __name__ == "__main__":
    text_classification()


Accuracy: 0.754
Enter a movie review: average
Sentiment: pos


In [ ]:
import nltk
import random
from nltk.corpus import movie_reviews

def extract_features(words):
    return dict([(word, True) for word in words])

def text_classification():
    nltk.download('movie_reviews', quiet=True)

    # Load movie reviews dataset
    documents = [(list(movie_reviews.words(fileid)), category)
                 for category in movie_reviews.categories()
                 for fileid in movie_reviews.fileids(category)]

    random.shuffle(documents)

    featuresets = [(extract_features(words), category) for (words, category) in documents]
    train_set, test_set = featuresets[:1500], featuresets[1500:]

    classifier = nltk.NaiveBayesClassifier.train(train_set)

    accuracy = nltk.classify.accuracy(classifier, test_set)
    print("Accuracy:", accuracy)

    # Multi-class classification: Define additional categories (optional)
    categories = movie_reviews.categories()

    user_input = input("Enter a review to classify: ")
    features = extract_features(user_input.split())

    sentiment = classifier.classify(features)
    print("Sentiment:", sentiment)

    # Print probabilities for all categories
    probabilities = classifier.prob_classify(features)
    for category in categories:
        print(f"Probability of '{category}': {probabilities.prob(category):.4f}")

if __name__ == "__main__":
    text_classification()


Accuracy: 0.722
Enter a review to classify: good
Sentiment: pos
Probability of 'neg': 0.4860
Probability of 'pos': 0.5140


In [ ]:
import nltk
import random
from nltk.corpus import movie_reviews
from sklearn.model_selection import KFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import make_pipeline
from sklearn import metrics

def extract_features(words):
    return dict([(word, True) for word in words])

def text_classification():
    nltk.download('movie_reviews', quiet=True)

    # Load movie reviews dataset
    documents = [(list(movie_reviews.words(fileid)), category)
                 for category in movie_reviews.categories()
                 for fileid in movie_reviews.fileids(category)]

    random.shuffle(documents)

    # Prepare the dataset
    sentences = [' '.join(words) for words, category in documents]
    labels = [category for words, category in documents]

    # Prepare cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    accuracy_scores = []

    for train_index, test_index in kf.split(sentences):
        X_train, X_test = [sentences[i] for i in train_index], [sentences[i] for i in test_index]
        y_train, y_test = [labels[i] for i in train_index], [labels[i] for i in test_index]

        # Create a pipeline with CountVectorizer and MultinomialNB
        model = make_pipeline(CountVectorizer(), MultinomialNB())
        model.fit(X_train, y_train)

        # Predict and evaluate
        y_pred = model.predict(X_test)
        accuracy = metrics.accuracy_score(y_test, y_pred)
        accuracy_scores.append(accuracy)

    print("Cross-Validation Accuracies:", accuracy_scores)
    print("Mean Accuracy:", sum(accuracy_scores) / len(accuracy_scores))

    # Input for single review classification
    user_input = input("Enter a review to classify: ")
    sentiment = model.predict([user_input])
    print("Sentiment:", sentiment[0])

if __name__ == "__main__":
    text_classification()


Cross-Validation Accuracies: [0.8225, 0.7925, 0.81, 0.8025, 0.835]
Mean Accuracy: 0.8125
Enter a review to classify: good
Sentiment: neg


In [ ]:
import nltk
import random
from nltk.corpus import movie_reviews
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

def create_feature_sets(documents):
    return [' '.join(words) for words, category in documents], [category for words, category in documents]

def text_classification():
    nltk.download('movie_reviews', quiet=True)

    # Load movie reviews dataset
    documents = [(list(movie_reviews.words(fileid)), category)
                 for category in movie_reviews.categories()
                 for fileid in movie_reviews.fileids(category)]

    random.shuffle(documents)

    # Prepare the dataset
    sentences, labels = create_feature_sets(documents)

    # Define classifiers
    classifiers = {
        "Naive Bayes": MultinomialNB(),
        "Decision Tree": DecisionTreeClassifier(),
        "Logistic Regression": LogisticRegression(max_iter=200)
    }

    # Prepare cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # Iterate through classifiers
    for classifier_name, classifier in classifiers.items():
        accuracy_scores = []

        for train_index, test_index in kf.split(sentences):
            X_train, X_test = [sentences[i] for i in train_index], [sentences[i] for i in test_index]
            y_train, y_test = [labels[i] for i in train_index], [labels[i] for i in test_index]

            # Create a pipeline with CountVectorizer
            model = make_pipeline(CountVectorizer(), classifier)
            model.fit(X_train, y_train)

            # Predict and evaluate
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            accuracy_scores.append(accuracy)

        print(f"{classifier_name} Cross-Validation Accuracies: {accuracy_scores}")
        print(f"{classifier_name} Mean Accuracy: {sum(accuracy_scores) / len(accuracy_scores)}")

    # Input for single review classification using the best classifier (example: Naive Bayes)
    user_input = input("Enter a review to classify using Naive Bayes: ")
    naive_bayes_model = make_pipeline(CountVectorizer(), MultinomialNB())
    naive_bayes_model.fit(sentences, labels)
    sentiment = naive_bayes_model.predict([user_input])
    print("Sentiment (Naive Bayes):", sentiment[0])

if __name__ == "__main__":
    text_classification()


if __name__ == "__main__":
    text_classification()


Naive Bayes Cross-Validation Accuracies: [0.82, 0.8025, 0.8375, 0.785, 0.805]
Naive Bayes Mean Accuracy: 0.8099999999999999
Decision Tree Cross-Validation Accuracies: [0.605, 0.665, 0.6325, 0.6075, 0.6225]
Decision Tree Mean Accuracy: 0.6265


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Logistic Regression Cross-Validation Accuracies: [0.8525, 0.83, 0.8325, 0.8225, 0.825]
Logistic Regression Mean Accuracy: 0.8325000000000001
